In [11]:
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import torch

# Cosine Similarity

In [12]:
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    dotProd = np.dot(vec1, vec2)
    normA = np.linalg.norm(vec1)
    normB = np.linalg.norm(vec2)
    return dotProd / (normA * normB)

# Neural Network

In [13]:
heart = pd.read_csv('heart.csv')

heart['Sex'] = (heart['Sex'] == 'M').astype(int)
heart['ExerciseAngina'] = (heart['ExerciseAngina'] == 'Y').astype(int)

heart_X = heart.drop(columns=['HeartDisease'])
heart_y = heart['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(heart_X, heart_y, test_size=0.2, random_state=42)

In [14]:
numerics = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
categoricals = ['ChestPainType', 'RestingECG', 'ST_Slope']
passthroughs = ['Sex', 'FastingBS', 'ExerciseAngina']

In [15]:
col_transform = make_column_transformer(
    (make_pipeline(StandardScaler()), numerics),
    (OneHotEncoder(), categoricals),
    ('passthrough', passthroughs )
)

X_train_transformed = col_transform.fit_transform(X_train)

column_names = col_transform.get_feature_names_out()

X_train_transformed_df = pd.DataFrame(X_train_transformed, columns=column_names, index=X_train.index)
X_train_transformed_df.columns = [name.split('__')[-1] for name in X_train_transformed_df.columns]

X_train_transformed_df.head()

,Age,RestingBP,Cholesterol,MaxHR,Oldpeak,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up,Sex,FastingBS,ExerciseAngina
795,-1.245067,-0.708985,0.372803,2.284353,-0.097061,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
25,-1.886236,-0.166285,0.086146,1.652241,-0.836286,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
84,0.250993,0.919115,0.123134,-0.441628,0.087745,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0
10,-1.779375,-0.166285,0.104640,0.229991,-0.836286,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
344,-0.283314,-0.708985,-1.846478,-1.271274,-0.836286,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0


In [16]:
X_test_transformed = col_transform.transform(X_test)

X_test_transformed_df = pd.DataFrame(X_test_transformed, columns=column_names, index=X_test.index)
X_test_transformed_df.columns = [name.split('__')[-1] for name in X_test_transformed_df.columns]

X_test_transformed_df.head()

,Age,RestingBP,Cholesterol,MaxHR,Oldpeak,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up,Sex,FastingBS,ExerciseAngina
668,0.999024,0.376415,-0.043312,1.691748,-0.836286,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
30,-0.069591,0.647765,2.943471,-0.244093,-0.836286,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
377,1.212747,1.461816,-1.846478,-0.560148,0.272552,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0
535,0.250993,-0.166285,-1.846478,-0.560148,0.087745,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
807,0.037270,-1.360226,1.010846,0.783088,-0.836286,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0


In [17]:
# Build the network
class HeartDiseaseNN(torch.nn.Module):
    def __init__(self, input_size):
        super(HeartDiseaseNN, self).__init__()
        self.fc1 = torch.nn.Linear(input_size, 36)
        self.fc2 = torch.nn.Linear(36, 12)
        self.fc3 = torch.nn.Linear(12, 2)
        self.relu = torch.nn.ReLU()
        self.dropout = torch.nn.Dropout(p=0.2)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

In [18]:
X_train, X_val, y_train, y_val = train_test_split(X_train_transformed_df, y_train, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
X_val_tensor = torch.tensor(X_val.values, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_transformed_df.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

In [19]:
model = HeartDiseaseNN(input_size=X_train.shape[1])
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 400
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_tensor)
            val_loss = criterion(val_outputs, y_val_tensor)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}')

Epoch [10/400], Loss: 0.6478, Val Loss: 0.6462
Epoch [20/400], Loss: 0.6126, Val Loss: 0.6019
Epoch [30/400], Loss: 0.5681, Val Loss: 0.5502
Epoch [40/400], Loss: 0.5228, Val Loss: 0.4907
Epoch [50/400], Loss: 0.4743, Val Loss: 0.4277
Epoch [60/400], Loss: 0.4252, Val Loss: 0.3747
Epoch [70/400], Loss: 0.3967, Val Loss: 0.3387
Epoch [80/400], Loss: 0.3898, Val Loss: 0.3172
Epoch [90/400], Loss: 0.3906, Val Loss: 0.3071
Epoch [100/400], Loss: 0.3781, Val Loss: 0.3011
Epoch [110/400], Loss: 0.3698, Val Loss: 0.2958
Epoch [120/400], Loss: 0.3454, Val Loss: 0.2909
Epoch [130/400], Loss: 0.3338, Val Loss: 0.2863
Epoch [140/400], Loss: 0.3348, Val Loss: 0.2826
Epoch [150/400], Loss: 0.3107, Val Loss: 0.2798
Epoch [160/400], Loss: 0.3132, Val Loss: 0.2785
Epoch [170/400], Loss: 0.3224, Val Loss: 0.2768
Epoch [180/400], Loss: 0.3258, Val Loss: 0.2750
Epoch [190/400], Loss: 0.3188, Val Loss: 0.2737
Epoch [200/400], Loss: 0.2916, Val Loss: 0.2718
Epoch [210/400], Loss: 0.3052, Val Loss: 0.2720
E

In [20]:
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    accuracy = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    print(f'Test Accuracy: {accuracy:.4f}')

Test Accuracy: 0.9076
